In [ ]:
!pip install -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

In [ ]:
import jax
import jax.numpy as jnp
import functools
import pickle
from dataclasses import dataclass
import dataclasses
import math
from jax.experimental.pallas.ops.tpu import flash_attention as fa

@dataclass
class BlockParams:
    W_q: jax.Array
    W_k: jax.Array
    W_v: jax.Array
    W_1: jax.Array
    W_2: jax.Array
    gamma1: jax.Array
    beta1: jax.Array
    gamma2: jax.Array
    beta2: jax.Array
    n_heads: int = 1
    embed_dim: int = 256
    hidden_dim: int = 1024

    @classmethod
    def random_init(cls, key: jax.Array, embed_dim: int, hidden_dim: int, n_heads: int) -> 'BlockParams':
        def init(key: jax.Array, shape: tuple[int, ...]) -> jax.Array:
            scale = 1.0/shape[0]
            return jax.random.normal(key, shape) * scale
        
        shapes = [
        (embed_dim, embed_dim),    # W_q
        (embed_dim, embed_dim),    # W_k
        (embed_dim, embed_dim),    # W_v
        (embed_dim, hidden_dim),   # W_1
        (hidden_dim, embed_dim),   # W_2
        ]
        keys = jax.random.split(key, len(shapes))
        W_q, W_k, W_v, W_1, W_2 = [init(k, s) for k, s in zip(keys, shapes)]
        return cls(
        W_q, W_k, W_v, W_1, W_2,
        gamma1=jnp.ones(embed_dim),  beta1=jnp.zeros(embed_dim),
        gamma2=jnp.ones(embed_dim),  beta2=jnp.zeros(embed_dim),
        n_heads=n_heads, embed_dim=embed_dim, hidden_dim=hidden_dim,
        )
        

jax.tree_util.register_dataclass(
    BlockParams,
    data_fields=['W_q', 'W_k', 'W_v', 'W_1', 'W_2',
                 'gamma1', 'beta1', 'gamma2', 'beta2'],
    meta_fields=['n_heads','embed_dim','hidden_dim'],
)

@jax.jit
def compute_qkv(X: jax.Array, W_q: jax.Array, W_k: jax.Array, W_v: jax.Array) -> tuple[jax.Array, jax.Array, jax.Array]:
    Q = jnp.dot(X, W_q)
    K = jnp.dot(X, W_k)
    V = jnp.dot(X, W_v)
    return Q, K, V

@jax.jit
def compute_attention(Q: jax.Array, K: jax.Array, V: jax.Array) -> jax.Array:
    d_k = Q.shape[-1]
    QK = jnp.matmul(Q, jnp.swapaxes(K, -1, -2)) / jnp.sqrt(d_k)
    attention_weights = jax.nn.softmax(QK)
    attention_output = jnp.matmul(attention_weights, V)
    return attention_output

@functools.partial(jax.jit, static_argnames=['n_heads'])
def split_heads(x: jax.Array, n_heads: int) -> jax.Array:
    *batch, seq, embed_dim = x.shape
    head_dim = embed_dim // n_heads
    x = x.reshape(*batch, seq, n_heads, head_dim)
    return jnp.swapaxes(x, -3, -2)

def merge_heads(x: jax.Array) -> jax.Array:
    x = jnp.swapaxes(x, -3, -2)   
    return x.reshape(*x.shape[:-2], -1)

@jax.jit
def compute_multi_head_attention(Q: jax.Array, K: jax.Array, V: jax.Array) -> jax.Array:
    """Reference attention. Q, K, V: (batch, heads, seq, head_dim) -- already split,
    already roped. Same contract as compute_flash_attention, so the two are swappable."""
    d_k = Q.shape[-1]

    QK = jnp.matmul(Q, jnp.swapaxes(K, -1, -2)) / jnp.sqrt(d_k)
    causal_mask = -jnp.inf*jnp.ones(QK.shape)
    causal_mask = jnp.triu(causal_mask, k=1) 
    attention_weights = jax.nn.softmax(QK+causal_mask)
    attention_output = jnp.matmul(attention_weights, V)
    return merge_heads(attention_output)

@jax.jit
def compute_flash_attention(Q, K, V):
    # Pallas TPU flash attention; sm_scale is required (no implicit 1/sqrt(d)).
    return merge_heads(fa.flash_attention(Q, K, V, causal=True,
                                          sm_scale=1.0 / math.sqrt(Q.shape[-1])))

def rope_tables(seq_len: int, dim: int, base: float = 10000.0) -> tuple[jax.Array, jax.Array]:
    """`dim` is the *per-head* width (head_dim), not embed_dim: RoPE is applied after
    the head split, so every head gets the same full spread of frequencies."""
    inv_freq = base ** (-jnp.arange(0, dim, 2) / dim)   # (dim/2,)
    angles   = jnp.outer(jnp.arange(seq_len), inv_freq)  # (seq, dim/2)
    return jnp.cos(angles), jnp.sin(angles)

@jax.jit
def apply_rope(x: jax.Array, cos: jax.Array, sin: jax.Array) -> jax.Array:
    # x: (..., seq, head_dim), pairs adjacent dims (0,1),(2,3),...
    # cos/sin: (seq, head_dim/2) -- broadcasts over any leading batch/head axes
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = jnp.stack([x1 * cos - x2 * sin,
                     x1 * sin + x2 * cos], axis=-1)
    return out.reshape(x.shape)

@jax.jit
def compute_masked_attention(Q: jax.Array, K: jax.Array, V: jax.Array) -> jax.Array:
    d_k = Q.shape[-1]
    QK = jnp.matmul(Q, jnp.swapaxes(K, -1, -2)) / jnp.sqrt(d_k)
    causal_mask = -jnp.inf*jnp.ones(QK.shape)
    causal_mask = jnp.triu(causal_mask, k=1) 
    attention_weights = jax.nn.softmax(QK + causal_mask)
    attention_output = jnp.matmul(attention_weights, V)
    return attention_output

@jax.jit
def layer_norm(X: jax.Array, gamma: jax.Array, beta: jax.Array) -> jax.Array:
    mean = jnp.mean(X, axis=-1, keepdims=True)
    var = jnp.var(X, axis=-1, keepdims=True)
    return gamma * (X - mean) / jnp.sqrt(var + 1e-5) + beta

@jax.jit
def relu(X: jax.Array) -> jax.Array:
    return jnp.maximum(0, X)

@jax.jit
def feed_forward(X: jax.Array, W_1: jax.Array, W_2: jax.Array) -> jax.Array:
    return jnp.dot(relu(jnp.dot(X, W_1)), W_2)

@jax.jit
def transformer_block(X: jax.Array, params: BlockParams) -> jax.Array:
    Q, K, V = compute_qkv(layer_norm(X, params.gamma1, params.beta1), params.W_q, params.W_k, params.W_v)
    seq = X.shape[-2]
    n_heads = params.n_heads

    Q = split_heads(Q, n_heads)   
    K = split_heads(K, n_heads)
    V = split_heads(V, n_heads)

    Q = apply_rope(Q, COS[:seq], SIN[:seq])
    K = apply_rope(K, COS[:seq], SIN[:seq])
    X = X + compute_flash_attention(Q, K, V)
    X = X + feed_forward(layer_norm(X, params.gamma2, params.beta2), params.W_1, params.W_2)
    return X

@dataclass
class ModelParams:
    W_embed: jax.Array
    blocks: list[BlockParams]
    W_decode: jax.Array
    vocab_size: int
    n_blocks: int 
    embed_dim: int 
    hidden_dim: int 

    @classmethod
    def random_init(cls, key: jax.Array, vocab_size: int, n_blocks: int, embed_dim: int, hidden_dim: int, n_heads: int, max_seq: int) -> tuple['ModelParams', tuple[jax.Array, jax.Array]]:
        k_embed, k_decode, k_blocks = jax.random.split(key, 3)
        W_embed  = jax.random.normal(k_embed,  (vocab_size, embed_dim)) / jnp.sqrt(embed_dim)
        W_decode = jax.random.normal(k_decode, (embed_dim, vocab_size)) / jnp.sqrt(embed_dim)
        blocks   = [BlockParams.random_init(k, embed_dim, hidden_dim, n_heads)
                    for k in jax.random.split(k_blocks, n_blocks)]

        return cls(W_embed, blocks, W_decode, vocab_size, n_blocks, embed_dim, hidden_dim), rope_tables(max_seq, embed_dim // n_heads)


jax.tree_util.register_dataclass(
    ModelParams,
    data_fields=['W_embed', 'blocks', 'W_decode'],
    meta_fields=['vocab_size', 'n_blocks', 'embed_dim', 'hidden_dim'],
)

@dataclass
class AdamWState:
    m: ModelParams
    v: ModelParams
    t: jax.Array
    b1: float = 0.9
    b2: float = 0.999
    eps: float = 1e-9
    wd: float = 0.01

    @classmethod
    def init(cls, params: ModelParams, **hparams: float) -> 'AdamWState':
        return cls(m=jax.tree.map(jnp.zeros_like, params),
              v=jax.tree.map(jnp.zeros_like, params),
              t=jnp.array(0, jnp.int32),
              **hparams,)

jax.tree_util.register_dataclass(
    AdamWState,
    data_fields=['m','v','t'],
    meta_fields=['b1','b2','eps','wd'],
)

@jax.jit
def model_forward(model_params: ModelParams, X: jax.Array) -> jax.Array:
    X = model_params.W_embed[X]
    for block in model_params.blocks:
        X = transformer_block(X, block)
    return X @ model_params.W_decode

@jax.jit
def cross_entropy_loss(logits: jax.Array, targets: jax.Array) -> jax.Array:
    logp = jax.nn.log_softmax(logits, -1)
    return -jnp.mean(jnp.take_along_axis(logp, targets[..., None], -1))

@jax.jit
def masked_cross_entropy_loss(logits: jax.Array, targets: jax.Array, mask: jax.Array) -> tuple[jax.Array, jax.Array]:
    logp = jax.nn.log_softmax(logits, -1)
    nll = -jnp.take_along_axis(logp, targets[..., None], -1).squeeze(-1) #selects target index across log probs
    nll = nll*mask
    count = mask.sum()*targets.shape[-1]
    return nll.sum()/(count), count

@jax.jit
def loss(params: ModelParams, X: jax.Array, targets: jax.Array) -> jax.Array:
    logits = model_forward(params, X)
    return cross_entropy_loss(logits, targets)

@jax.jit
def masked_loss(params: ModelParams, X: jax.Array, targets: jax.Array, mask: jax.Array) -> tuple[jax.Array, jax.Array]:
    logits = model_forward(params, X)
    loss, count = masked_cross_entropy_loss(logits, targets, mask)
    return (loss, count)

@jax.jit
def train_step(params: ModelParams, X: jax.Array, Y: jax.Array, mask: jax.Array, optim_state: AdamWState, lr: jax.Array) -> tuple[tuple[ModelParams, AdamWState], tuple[jax.Array, jax.Array]]:
    (loss_val, count), grads = jax.value_and_grad(masked_loss, has_aux=True)(params, X, Y, mask)
    params, optim_state = AdamW_optim_step(params, grads, optim_state, lr)
    return (params, optim_state), (loss_val, count)

@jax.jit
def train_batch_step(carry: tuple[ModelParams, AdamWState, jax.Array], xs: tuple[jax.Array, jax.Array]) -> tuple[tuple[ModelParams, AdamWState, jax.Array], tuple[jax.Array, jax.Array]]:
    params, optim_state, lr = carry
    batches, mask = xs
    X, Y = batches[:, :-1], batches[:, 1:]   
    (params, optim_state), out = train_step(params, X, Y, mask, optim_state, lr)
    return (params, optim_state, lr), out

@jax.jit
def train_epoch_step(params: ModelParams, optim_state: AdamWState, batches: jax.Array, masks: jax.Array, lr: jax.Array) -> tuple[ModelParams, AdamWState, jax.Array]:
    (params, optim_state, lr), (batch_losses, batch_counts) = jax.lax.scan(
        train_batch_step, (params, optim_state, lr), (batches, masks))
    epoch_loss = jnp.sum(batch_losses * batch_counts) / jnp.sum(batch_counts)   # exact 
    return params, optim_state, epoch_loss

@jax.jit
def valid_step(params: ModelParams, X: jax.Array, Y: jax.Array, mask: jax.Array) -> tuple[jax.Array, jax.Array]:
    loss_val, count = masked_loss(params, X, Y, mask)
    return loss_val, count

@jax.jit
def valid_batch_step(params: ModelParams, xs: tuple[jax.Array, jax.Array]) -> tuple[ModelParams, tuple[jax.Array, jax.Array]]:
    batches, mask = xs
    X, Y = batches[:, :-1], batches[:, 1:]   
    loss_val, count = valid_step(params, X, Y, mask)
    return params, (loss_val, count) 

@jax.jit
def valid_epoch_step(params: ModelParams, batches: jax.Array, masks: jax.Array) -> jax.Array:
    step = lambda p, xs: valid_batch_step(p, xs)
    params, (batch_losses, batch_counts) = jax.lax.scan(step, params, (batches, masks))
    epoch_loss = jnp.sum(batch_losses * batch_counts) / jnp.sum(batch_counts)  
    return epoch_loss


@dataclass
class AdamWState:
    m: ModelParams
    v: ModelParams
    t: jax.Array
    b1: float = 0.9
    b2: float = 0.999
    eps: float = 1e-9
    wd: float = 0.01

    @classmethod
    def init(cls, params: ModelParams, **hparams: float) -> 'AdamWState':
        return cls(m=jax.tree.map(jnp.zeros_like, params),
              v=jax.tree.map(jnp.zeros_like, params),
              t=jnp.array(0, jnp.int32),
              **hparams,)

jax.tree_util.register_dataclass(
    AdamWState,
    data_fields=['m','v','t'],
    meta_fields=['b1','b2','eps','wd'],
)

@jax.jit
def AdamW_optim_step(params: ModelParams, grad: ModelParams, optim_state: AdamWState, lr: jax.Array) -> tuple[ModelParams, AdamWState]:
    _m,_v,_t = optim_state.m, optim_state.v, optim_state.t
    b1, b2, eps, wd = optim_state.b1, optim_state.b2, optim_state.eps, optim_state.wd

    t = _t + 1
    m = jax.tree.map(lambda _m, g: b1*_m + (1-b1)*g, _m, grad)
    v = jax.tree.map(lambda _v, g: b2*_v + (1-b2)*g**2, _v, grad)

    bc1 = 1 - b1**t
    bc2 = 1 - b2**t
    mhat = jax.tree.map(lambda m: m / bc1, m)
    vhat = jax.tree.map(lambda v: v / bc2, v)

    def weight_update(params: jax.Array, mhat: jax.Array, vhat: jax.Array, eps: float, wd: float, lr: jax.Array) -> jax.Array:
        return params - lr * (mhat / (jnp.sqrt(vhat)+eps) + wd*params)

    f = lambda p, m, v: weight_update(p, m, v, eps, wd, lr)
    params =  jax.tree.map(f, params, mhat, vhat) 
    return params, dataclasses.replace(optim_state, m=m, v=v, t=t)

@jax.jit
def cosine_lr_schudule(t: jax.Array | int, peak: float, warmup: int, end: float, total: int) -> jax.Array:
    warm  = peak * t / warmup
    prog  = jnp.clip((t - warmup) / (total - warmup), 0.0, 1.0)
    cos   = 0.5 * (1.0 + jnp.cos(jnp.pi * prog))
    decay = end + (peak - end) * cos
    return jnp.where(t < warmup, warm, decay)  


EMBED_DIM    = 1024
ROPE_MAX_SEQ = 512

key = jax.random.key(42)
tokenizer = {c: i for i, c in enumerate("0123456789+= ")}
params, (COS, SIN) = ModelParams.random_init(key, vocab_size=len(tokenizer), n_blocks=16,
                                             embed_dim=EMBED_DIM, hidden_dim=1024, n_heads=2,
                                             max_seq=ROPE_MAX_SEQ)

X = jax.random.randint(key, (2, ROPE_MAX_SEQ), 0, len(tokenizer))
logits = model_forward(params, X)
print(logits.shape)
probs = jax.nn.softmax(logits, axis=-1)        
preds = jnp.argmax(logits, axis=-1)
print(preds)

# Pallas flash attention matches the naive reference
Q, K, V = jax.random.normal(key, (3, 2, 8, ROPE_MAX_SEQ, EMBED_DIM // 8))
assert jnp.allclose(compute_flash_attention(Q, K, V),
                    compute_multi_head_attention(Q, K, V), atol=1e-2)

optim_state = AdamWState.init(params)

print(optim_state)

(2, 512, 13)
[[10 10 10 ...  9  9  9]
 [12  0  0 ... 12 12 12]]
AdamWState(m=ModelParams(W_embed=Array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32), blocks=[BlockParams(W_q=Array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32), W_k=Array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32), W_v=Array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ...,

In [3]:
# FineWeb-EDU: nanochat's pretraining corpus (karpathy/fineweb-edu-100b-shuffle).
# Pre-shuffled parquet shards, ~100MB / ~250M chars each; one shard is plenty here.
from huggingface_hub import hf_hub_download
import pandas as pd

shard_path = hf_hub_download(
    repo_id="karpathy/fineweb-edu-100b-shuffle",
    filename="shard_00000.parquet",
    repo_type="dataset",
)
df = pd.read_parquet(shard_path)
texts = df["text"].tolist()
print(len(texts), "documents,", sum(len(t) for t in texts), "chars")
print(texts[0][:500])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


shard_00000.parquet: reconstructing file:   0%|          |  0.00B / 94.5MB            

shard_00000.parquet: downloading bytes:           |  0.00B            

53248 documents, 253426213 chars
Shipment & Transport-Sea, Air, Rail, Road, Pipeline
The mode of transportation is an important consideration when planning the shipment process. Besides the costs, the urgency of the shipment, the value of the goods being shipped as well as the size and weight of the goods need to be evaluated when determining the form of transportation.
Seaborne trade accounts for about 90% of the global trade, and as per UNCTAD, 1687 million tons (2015 estimate) were carried in around 177.6 million containers 


In [4]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=4096, special_tokens=["<pad>", "<unk>", "<s>", "</s>"])
tokenizer.train_from_iterator(texts, trainer=trainer)
tokenizer.decoder = decoders.ByteLevel()
tokenizer.save("tokenizer-4096.json")

print(tokenizer)


Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[{"id":0, "content":"<pad>", "single_word":False, "lstrip":False, "rstrip":False, ...}, {"id":1, "content":"<unk>", "single_word":False, "lstrip":False, "rstrip":False, ...}, {"id":2, "content":"<s>", "single_word":False, "lstrip":False, "rstrip":False, ...}, {"id":3, "content":"</s>", "single_word":False, "lstrip":False, "rstrip":False, ...}], normalizer=None, pre_tokenizer=ByteLevel(add_prefix_space=True, trim_offsets=True, use_regex=True), post_processor=None, decoder=ByteLevel(add_prefix_space=True, trim_offsets=True, use_regex=True), model=BPE(dropout=None, unk_token=None, continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={"<pad>":0, "<unk>":1, "<s>":2, "</s>":3, "!":4, ...}, merges=[("Ġ", "t"), ("Ġ", "a"), ("h", "e"), ("i", "n"), ("r", "e"), ...]))


In [5]:
data = []
tail = []
for text in texts:
    ids = tail + tokenizer.encode(text).ids
    tail = []  # reset before processing this text's chunks
    for i in range(0, len(ids), ROPE_MAX_SEQ):
        chunk = ids[i:i+ROPE_MAX_SEQ+1]
        if len(chunk) < ROPE_MAX_SEQ+1:
            tail = chunk
        else:
            data.append(chunk)

    
print(len(data))
print(texts[0][:500])
print(data[0])
print(tokenizer.decode(data[0]))

144281
Shipment & Transport-Sea, Air, Rail, Road, Pipeline
The mode of transportation is an important consideration when planning the shipment process. Besides the costs, the urgency of the shipment, the value of the goods being shipped as well as the size and weight of the goods need to be evaluated when determining the form of transportation.
Seaborne trade accounts for about 90% of the global trade, and as per UNCTAD, 1687 million tons (2015 estimate) were carried in around 177.6 million containers 
[280, 1002, 317, 1298, 3651, 1920, 16, 54, 72, 68, 15, 277, 289, 15, 370, 585, 15, 3865, 296, 15, 318, 469, 293, 415, 171, 417, 824, 72, 232, 3155, 266, 261, 302, 941, 1710, 266, 588, 3988, 212, 392, 3243, 903, 17, 327, 220, 1072, 212, 3214, 15, 212, 3465, 3026, 232, 212, 392, 3243, 15, 212, 2051, 232, 212, 1008, 86, 915, 392, 1876, 228, 295, 760, 295, 212, 2149, 238, 2349, 232, 212, 1008, 86, 665, 239, 260, 2995, 413, 588, 1675, 1382, 212, 820, 232, 3155, 266, 17, 171, 54, 72, 2336, 736

In [6]:
import random
random.seed(42)
random.shuffle(data)

n = len(data)
test_frac, valid_frac = 0.1, 0.1
test_slice  = int(n * test_frac)
valid_slice = int(n * valid_frac)

test  = jnp.array(data[:test_slice])
valid = jnp.array(data[test_slice:test_slice + valid_slice])
train = jnp.array(data[test_slice + valid_slice:])

batch_size = 64   # seq_len=ROPE_MAX_SEQ -> ~33k tokens/step; 256 OOMs (activation-bound, ~35G)

print(len(train), len(test), len(valid))
print("batch_size", batch_size, "tokens/step", batch_size * ROPE_MAX_SEQ)

115425 14428 14428
batch_size 64 tokens/step 32768


In [8]:
import jax, shutil
import os
from google.colab import drive
drive.mount('/content/drive')

DB_LOCAL = "/content/mlflow.db"                    # fast, reliable VM-local disk
DB_DRIVE = "/content/drive/MyDrive/mlflow.db"      # synced copy your laptop reads
if os.path.exists(DB_DRIVE):                       # resume existing history if present
    shutil.copy(DB_DRIVE, DB_LOCAL)

Mounted at /content/drive


In [9]:
params, (COS, SIN) = ModelParams.random_init(key, vocab_size=tokenizer.get_vocab_size(), n_blocks=10,
                                             embed_dim=1024, hidden_dim=4*1024, n_heads=8,
                                             max_seq=ROPE_MAX_SEQ)

optim_state = AdamWState.init(params)

peak = 1e-5
warmup = 1000
end = 3e-7
total = 20


# roofline: peak HBM and per-step FLOPs are per-batch, so a few batches profile the
# same as a full epoch -- no padding needed, we just take an exact multiple.
n_train_batches = 8
train_batches = train[:n_train_batches*batch_size].reshape(n_train_batches, batch_size, -1)
train_masks   = jnp.ones((n_train_batches, batch_size, 1))

#first jit compilation
print("first")
i=1
lr = cosine_lr_schudule(i, peak, warmup, end, total)
params, optim_state, epoch_loss = train_epoch_step(params, optim_state, train_batches, train_masks, lr)
epoch_loss.block_until_ready()
print("finished")
jax.profiler.start_trace("/content/jax-trace")
lr = cosine_lr_schudule(i, peak, warmup, end, total)
params, optim_state, epoch_loss = train_epoch_step(params, optim_state, train_batches, train_masks, lr)
epoch_loss.block_until_ready()
jax.profiler.stop_trace()

shutil.make_archive("/content/jax-trace", "gztar", "/content/jax-trace")
shutil.copy("/content/jax-trace.tar.gz", "/content/drive/MyDrive/jax-trace.tar.gz")


first
finished


'/content/drive/MyDrive/jax-trace.tar.gz'

In [33]:
epochs = 1000
from tqdm import tqdm
import mlflow, datetime, shutil, time

# DB_LOCAL / DB_DRIVE and the Drive mount come from the setup cell above.
mlflow.set_tracking_uri(f"sqlite:///{DB_LOCAL}")
mlflow.config.enable_async_logging()

# batch_size and ROPE_MAX_SEQ come from the data cells -- don't redefine them here,
# the chunking in cell 3 is keyed to ROPE_MAX_SEQ and batch_size is set by what fits.
key = jax.random.key(42)
params, (COS, SIN) = ModelParams.random_init(key, vocab_size=tokenizer.get_vocab_size(),
                                             n_blocks=10, embed_dim=1024, hidden_dim=4*1024,
                                             n_heads=8, max_seq=ROPE_MAX_SEQ)
optim_state = AdamWState.init(params)
n_heads = params.blocks[0].n_heads

mlflow.set_experiment(f"fineweb-edu-{params.n_blocks}L-{params.embed_dim}d-{n_heads}h")

peak = 3e-4
warmup = 50
end = 3e-7
total = epochs

if len(train) % batch_size != 0:
    tail_size = -len(train) % batch_size
    train_padded = jnp.pad(train, ((0, tail_size),(0,0)))
    train_mask = jnp.ones(len(train_padded)).at[len(train_padded)-tail_size:].set(0)
else:
    train_mask = jnp.ones(len(train))
    train_padded = train

if len(valid) % batch_size != 0:
    tail_size = -len(valid) % batch_size
    valid_padded = jnp.pad(valid, ((0, tail_size),(0,0)))
    valid_mask = jnp.ones(len(valid_padded)).at[len(valid_padded)-tail_size:].set(0)
else:
    valid_mask = jnp.ones(len(valid))
    valid_padded = valid

n_train_batches = len(train_padded)//batch_size

train_batches = train_padded[:n_train_batches*batch_size].reshape(n_train_batches, batch_size, -1)
train_masks = train_mask[:n_train_batches*batch_size].reshape(n_train_batches, batch_size, -1)

n_valid_batches = len(valid_padded)//batch_size

valid_batches = valid_padded[:n_valid_batches*batch_size].reshape(n_valid_batches, batch_size, -1)
valid_masks = valid_mask[:n_valid_batches*batch_size].reshape(n_valid_batches, batch_size, -1)

print("train batches", train_batches.shape,"mask", train_masks.shape)
print("valid batches", valid_batches.shape,"mask", valid_masks.shape)

run_name = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
dev = jax.devices()[0]
buf = []

with mlflow.start_run(run_name=run_name):
    mlflow.log_params({"epochs": epochs, "batch_size": batch_size, "seq_len": ROPE_MAX_SEQ,
                       "vocab_size": params.vocab_size, "n_blocks": params.n_blocks,
                       "embed_dim": params.embed_dim, "hidden_dim": params.hidden_dim,
                       "n_heads": n_heads,
                       "lr_peak": peak, "lr_end": end, "warmup": warmup})

    n_examples = train_batches.shape[0] * train_batches.shape[1]
    t_window = time.perf_counter()   # start of the current 25-epoch logging window
    for i in tqdm(range(epochs), total=epochs):
        lr = cosine_lr_schudule(i, peak, warmup, end, total)
        params, optim_state, epoch_loss = train_epoch_step(params, optim_state, train_batches, train_masks, lr)
        valid_loss = valid_epoch_step(params, valid_batches, valid_masks)

        buf.append((i, epoch_loss, valid_loss))
        if i %25 == 0:
            steps, train_losses, valid_losses = zip(*buf); buf = []
            # first real sync point: this drains every epoch queued since the last drain
            train_losses = jax.device_get(jnp.stack(train_losses))
            valid_losses = jax.device_get(jnp.stack(valid_losses))

            for s, tl, vl in zip(steps, train_losses, valid_losses):
                mlflow.log_metrics({
                    "train_loss":     float(tl),
                    "valid_loss":     float(vl),
                }, step=s)
            mem = dev.memory_stats()
            dt  = time.perf_counter() - t_window     # wall time for the whole window...
            t_window = time.perf_counter()
            mlflow.log_metrics({
                "lr":               float(lr),
                "examples_per_sec": n_examples * len(steps) / dt,   # ...so divide by its epochs
                "tokens_per_sec":   n_examples * len(steps) * ROPE_MAX_SEQ / dt,
                "tpu_mem_used_gb":  mem["bytes_in_use"]      / 1e9,
                "tpu_mem_peak_gb":  mem["peak_bytes_in_use"] / 1e9,
            }, step=i)
    ckpt_path = "/content/model_params.pkl"
    with open(ckpt_path, "wb") as f:
        pickle.dump(jax.device_get(params), f)
    mlflow.log_artifact(ckpt_path, artifact_path="checkpoints")
    shutil.copy(ckpt_path, "/content/drive/MyDrive/model_params.pkl")

shutil.copy(DB_LOCAL, DB_DRIVE)                     # push updated db back to Drive (paths from setup cell)
print("synced ->", DB_DRIVE)


In [ ]:
print(jax.devices())